# M1과 M3+M4의 공통 누락·신규 적중·손실 진단
기존 Dunnhumby 개발구간 DAY684–690, seed43·44, 300epoch checkpoint만 읽습니다. 새 학습·모델 선택·마지막 주 test 없음. 현재 학습 중인 노트북과 별도 CPU 런타임에서 실행하세요.

필수 Drive 자료: `results_v3_dunnhumby_clv_m2_capacity_search_v1`의 M1 baseline 두 시드 결과/checkpoint, 기존 M3 결과 JSON, `results_v3_dunnhumby_clv_m5_m3_m4_split_nv_s43_v1` 및 s44의 결과·checkpoint, 원래 Dunnhumby 입력 데이터. 없거나 출처가 다르면 중단합니다.

전체 142지표 재현 후 정답 Top100 순위·Top50 추천을 저장합니다. 100위 밖 순위는 결측으로 표시합니다. 상품 구매자 수·금액 백분위·개인 카테고리 거래/지출 비중·가격위치 거리는 TRAIN만 사용합니다. 공통 누락과 오추천의 특징을 동일 고객 안에서 비교하며 인과·유의성·새 모델 성공을 주장하지 않습니다.

로컬 최소검사는 통과했지만 실제 Drive checkpoint를 이용한 전체 실행은 아직 검증되지 않았습니다. 아래 셀을 위에서부터 실행하고 ZIP을 전달하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
SOURCE_COMMIT = '87459b39958dd92abe696a8763879b11693252f5'
repo = Path('/content/clv-m1-m5-diagnostic-' + SOURCE_COMMIT[:12])
if not repo.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(repo))


## 기존 모델 읽기·검증·진단
CPU로 두 시드의 M1과 M3+M4를 읽습니다. 기존 결과와 재현 차이가 있으면 readback.csv를 남기고 중단합니다. 완료 전 셀을 중단하면 재실행 시 진단 계산은 처음부터 수행됩니다.

In [ ]:
import clv_m1_m5_error_diagnostic as diagnostic
paths = diagnostic.run()
print(json.dumps(paths, ensure_ascii=False, indent=2))


## 결과 확인·원본 다운로드
평균 비교는 동일 seed·세그먼트 안에서 읽습니다. item_buyers는 진단용 인기도 변수이며 CLV 제안기법의 입력으로 확정된 것이 아닙니다. feature_summary는 후보쌍 평균과 고객평균을 모두 포함하며, paired_user_feature_differences는 두 집단을 모두 가진 고객만 포함합니다. 이 표는 M1 점수·인기도를 맞춘 조건부 구별력 검사는 아닙니다.

In [ ]:
import pandas as pd
from IPython.display import display
summary = pd.read_csv(paths['summary'])
display(summary[summary.grouping.isin(['all','segment'])])
features = pd.read_csv(paths['feature_summary'])
display(features[(features.k == 10) & (features.segment == '고CLV')])
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive_path = Path('/content/clv_m1_m5_error_diagnostic_seed43_44.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(archive_path))
